# Point Cloud Combination — Full Example

A working sample going over the full process from start to finish.

The combination process is aimed at complementing existing large datasets with newer smaller datasets.
It also aims to leave as much of the original dataset intact as possible, assuming it is more detailed and precise.

The algorithm runs in two phases:
- **Removal phase**: out-of-date points in the original geometry are removed
- **Addition phase**: only new/changed points from the new geometry are added

## Imports and setup

In [ ]:
import numpy as np
import open3d as o3d
from pathlib import Path
import trimesh
import sys
sys.path.insert(0, '../')
import drm
from drm.combine import (
    combine_geometry,
    get_points_in_hull,
    filter_pcd_by_distance,
    get_invisible_points_grid,
    build_occlusion_grid
)


DATASET_DIR = Path(r"../../../datasets/V-Scan/data")
FOLDER_NAME = "bedroom_Leica-P30_1775809921180"
REFERENCE_NAME = "main.txt"
EMPTY_SCENE_NAME = "main_empty.txt"
VARIATION_GLOB = "main_var_*.txt"
VOXEL_SIZE = 0.05
# Distance threshold: the max coverage distance (metres)
THRESHOLD_RESOLUTION = 0.05

%load_ext autoreload
%autoreload 2


## Import the Data


In [ ]:
# Load the pointclouds from the folder
dataset_dir = Path(DATASET_DIR)
ref_path = dataset_dir / FOLDER_NAME / REFERENCE_NAME
empty_scene_path = dataset_dir / FOLDER_NAME / EMPTY_SCENE_NAME
var_paths = sorted((dataset_dir/FOLDER_NAME).glob(VARIATION_GLOB))
refPcd,_ = drm.txt_pcd_to_open3d(ref_path, apply_unity_conversion=True)
refPcd = refPcd.voxel_down_sample(VOXEL_SIZE)
emptyPcd,_ = drm.txt_pcd_to_open3d(empty_scene_path, apply_unity_conversion=True)
emptyPcd = emptyPcd.voxel_down_sample(VOXEL_SIZE)
ref_scan_pos = drm.read_transform_matrix(ref_path, apply_unity_conversion=True)[:3,3]

varPcds = []
varPosses = []
for var_path in var_paths:
    varPcd,_ = drm.txt_pcd_to_open3d(var_path, apply_unity_conversion=True)
    varPcd = varPcd.voxel_down_sample(VOXEL_SIZE)
    posTransform = drm.read_transform_matrix(var_path, apply_unity_conversion=True)[:3,3]  # just to check that the transform matrix is read correctly (it is)
    varPcds.append(varPcd)
    varPosses.append(posTransform)

In [ ]:
partialVar,_ ,_= drm.radial_fov_crop_pointcloud(varPcds[0], horizontal_fov_deg=180, vertical_fov_deg=180,  origin=varPosses[0])

In [ ]:
# Optionally visualise them
newScene = drm.visualise_open3d([refPcd, partialVar], random_color=True)
newScene.add_geometry(trimesh.creation.axis(origin_size=0.05, axis_length=1.0, axis_radius=0.005))
newScene.show()

---
## Removal Phase

### Step 1 — Convex hull of newGeometry

To prevent false removal of the original geometry, we limit evaluation to only the points of `ogGeometry` that fall within the spatial extent of `newGeometry`. A convex hull encapsulates this region.

In [ ]:
newGeoHull, _ = partialVar.compute_convex_hull()
newGeoHull.compute_triangle_normals()

scene = trimesh.Scene()

# Add point cloud
cloud = drm.o3d_pointcloud_to_trimesh(refPcd)
scene.add_geometry(cloud, node_name="cloud")

# Add lineset (e.g. a convex hull wireframe)
hull_lineset = o3d.geometry.LineSet.create_from_triangle_mesh(newGeoHull)
path = drm.lineset_to_trimesh_path(hull_lineset)
scene.add_geometry(path, node_name="hull")

scene.show()

### Step 2 — Filter irrelevant ogGeometry points

A subselection of the original geometry is made using the convex hull as a bounding volume.
Points outside the hull are considered irrelevant to this update and are kept unchanged.

In [ ]:
relevantOg, irrelevantOg = get_points_in_hull(refPcd, newGeoHull)

print(f"Relevant points   : {len(relevantOg.points)}")
print(f"Irrelevant points : {len(irrelevantOg.points)}")

relevantOg.paint_uniform_color([0.2, 0.8, 0.2])     # green = relevant
irrelevantOg.paint_uniform_color([0.6, 0.6, 0.6])   # grey  = irrelevant
drm.visualise_open3d([relevantOg, partialVar], random_color=True).show()

### Step 3 — Coverage check

We sample a dense point cloud from `newGeometry` and check which points of `relevantOg` are covered (close enough) vs uncovered (too far away).

Uncovered points are candidates for removal — they are either out-of-date, or were not visible to the new scanner.

In [ ]:
coveredPoints, unCoveredPoints = filter_pcd_by_distance(relevantOg, partialVar, THRESHOLD_RESOLUTION)

print(f"Covered points   : {len(coveredPoints.points)}")
print(f"Uncovered points : {len(unCoveredPoints.points)}")

coveredPoints.paint_uniform_color([0.2, 0.8, 0.2])    # green = still valid
unCoveredPoints.paint_uniform_color([1.0, 0.5, 0.0])  # orange = candidates for removal
drm.visualise_open3d([unCoveredPoints, coveredPoints]).show()

### Step 4 — Visibility check

Uncovered points are checked against the new mesh. Assuming the new scanner captured everything it could see:
- Points **inside** the mesh were not visible during scanning → keep them
- Points **outside** the mesh could have been seen → they are out-of-date → remove them

The check works by finding the closest mesh face to each point and comparing the point-to-face direction with the face normal.

In [ ]:
occupied_vox, occluded_vox = build_occlusion_grid(partialVar, varPosses[0], voxel_size=VOXEL_SIZE*2)
drm.visualise_open3d(drm.combine.visualise_occlusion_grid(occupied_vox, occluded_vox, varPosses[0], voxel_size=VOXEL_SIZE*2)).show()

In [ ]:
invisiblePoints, visiblePoints = get_invisible_points_grid(
    unCoveredPoints,
    partialVar,
    scanner_pos= varPosses[0],
    voxel_size=THRESHOLD_RESOLUTION
)

print(f"Invisible (kept)   : {len(invisiblePoints.points)}")
print(f"Visible (removed)  : {len(visiblePoints.points)}")

invisiblePoints.paint_uniform_color([0.0, 0.6, 1.0])  # blue  = kept (were hidden)
visiblePoints.paint_uniform_color([1.0, 0.2, 0.2])    # red   = removed (out-of-date)
drm.visualise_open3d([invisiblePoints, visiblePoints]).show()

---
## Addition Phase

### Step 5 — Filter new geometry to changed points only

Because we assume the original geometry is of better quality, we only add points from the new geometry that are genuinely new — i.e. not already covered by the original scan.
This is an inverted distance query from the new points back to the existing geometry.

In [ ]:
existingNewGeo, newNewGeo = filter_pcd_by_distance(partialVar, relevantOg, THRESHOLD_RESOLUTION)

print(f"Already covered by original : {len(existingNewGeo.points)}")
print(f"Genuinely new points        : {len(newNewGeo.points)}")

existingNewGeo.paint_uniform_color([0.6, 0.6, 0.6])  # grey  = already exists
newNewGeo.paint_uniform_color([0.0, 1.0, 0.4])       # green = truly new
drm.visualise_open3d([existingNewGeo, newNewGeo]).show()

### Step 6 — Combine everything

The final combined geometry consists of:
- `irrelevantOg` — original points outside the hull (untouched)
- `coveredPoints` — original points still covered by the new scan (valid, kept)
- `invisiblePoints` — original points not covered but hidden inside the mesh (kept)
- `newNewGeo` — genuinely new points from the new scan (added)

In [ ]:
newCombinedGeometry = irrelevantOg + coveredPoints + newNewGeo + invisiblePoints

print(f"irrelevantOg     : {len(irrelevantOg.points)}")
print(f"coveredPoints    : {len(coveredPoints.points)}")
print(f"invisiblePoints  : {len(invisiblePoints.points)}")
print(f"newNewGeo        : {len(newNewGeo.points)}")
print(f"─────────────────────────────────")
print(f"Combined total   : {len(newCombinedGeometry.points)}")

drm.visualise_open3d([newCombinedGeometry], random_color=True).show()

---
## Single-function shortcut

`combine_geometry()` runs all six steps above in one call and returns the combined geometry.

In [ ]:
# Re-create fresh inputs (no painted colors from above)


combinedGeometry = combine_geometry(emptyPcd,varPcds[1], posTransform[1], THRESHOLD_RESOLUTION, checkVisibility=True, logProcess=False)
print(f"\nFinal point count: {len(combinedGeometry.points)}")

drm.visualise_open3d([combinedGeometry]).show()

## Evaluation

In [ ]:
import re
from collections import defaultdict
import pandas as pd
import numpy as np
from pathlib import Path


# ── helpers ──────────────────────────────────────────────────────────────────

def coverage_ratio(source_pcd, target_pcd, threshold):
    """Fraction of *source* points that are within `threshold` of *target*."""
    dists = np.asarray(source_pcd.compute_point_cloud_distance(target_pcd))
    return float((dists < threshold).mean())


def env_type(folder_name: str) -> str:
    """
    Extract the environment-type prefix from a folder name of the form
    'environment-type_xxxxxx'.  Returns the full name as fallback.
    """
    m = re.match(r"^(.+)_\d+$", folder_name)
    return m.group(1) if m else folder_name


# ── main evaluation ───────────────────────────────────────────────────────────

def evaluate_pipeline_vs_replace(
    dataset_dir,
    reference_name   = "main.txt",
    empty_scene_name = "main_empty.txt",
    variation_glob   = "main_var_2.txt",   # only var_2 to get a different position and furniture
    voxel_size       = 0.05,
    threshold        = 0.05,
    check_visibility = True,
    max_scans         = None,  # set to an integer to limit the number of scans processed (for quick testing)
):
    """
    For every subfolder in `dataset_dir`:
      1. Load main.txt  (reference / original pointcloud)
      2. Load main_empty.txt (empty scene)
      3. Load main_var_2.txt (changed furniture & position)
      4. Subsample all to `voxel_size`
      5. Run combine_geometry  → *pipeline* result
      6. Build *replace* result  = empty_pcd + var_pcd  (outright replacement)
      7. Compute per-scan metrics and aggregate by environment type.

    Metrics
    -------
    points_kept_pct      : % of original (main) points still present in pipeline result
    points_updated_pct   : % of original points that were swapped/replaced (1 - kept)
    pipeline_coverage_gain : coverage(pipeline → empty) − coverage(var → empty)
                             i.e. how much MORE of the empty scene the pipeline covers
                             compared to the raw variation alone
    replace_coverage_gain  : same but for the outright-replace result
    """
    dataset_dir = Path(dataset_dir)
    rows = []

    subfolders = sorted(p for p in dataset_dir.iterdir() if p.is_dir())

    if max_scans is not None:
        subfolders = subfolders[:max_scans]

    for folder in subfolders:
        ref_path   = folder / reference_name
        empty_path = folder / empty_scene_name
        var_paths  = sorted(folder.glob(variation_glob))

        if not ref_path.exists() or not empty_path.exists() or not var_paths:
            print(f"[skip] {folder.name}  — missing required files")
            continue

        print(f"\n── {folder.name} ──")

        # ── load & subsample ──────────────────────────────────────────────
        ref_pcd,   _ = drm.txt_pcd_to_open3d(ref_path,   apply_unity_conversion=True)
        ref_pcd      = ref_pcd.voxel_down_sample(voxel_size)

        empty_pcd, _ = drm.txt_pcd_to_open3d(empty_path, apply_unity_conversion=True)
        empty_pcd    = empty_pcd.voxel_down_sample(voxel_size)

        for var_path in var_paths:
            var_pcd, _ = drm.txt_pcd_to_open3d(var_path, apply_unity_conversion=True)
            var_pcd    = var_pcd.voxel_down_sample(voxel_size)
            scanner_pos = drm.read_transform_matrix(var_path, apply_unity_conversion=True)[:3, 3]

            n_ref = len(ref_pcd.points)

            # ── pipeline result ───────────────────────────────────────────
            pipeline_pcd = combine_geometry(
                empty_pcd,
                var_pcd,
                scanner_pos,
                distanceTreshold=threshold,
                checkVisibility=check_visibility,
                logProcess=False,
            )
            n_pipeline = len(pipeline_pcd.points)

            # ── outright-replace result ───────────────────────────────────
            # Simply use the variation scan as-is (no combination logic)
            replace_pcd = var_pcd

            # ── points kept / updated ─────────────────────────────────────
            # "kept" = original points that survived into the pipeline result
            d_ref_to_pipeline = np.asarray(ref_pcd.compute_point_cloud_distance(pipeline_pcd))
            points_kept_pct   = float((d_ref_to_pipeline < threshold).mean()) * 100
            points_updated_pct = 100.0 - points_kept_pct

            # ── coverage gain over empty scene ────────────────────────────
            # baseline: how much of empty the raw variation already covers
            var_cov_empty      = coverage_ratio(empty_pcd, var_pcd,      threshold)
            pipeline_cov_empty = coverage_ratio(empty_pcd, pipeline_pcd, threshold)
            replace_cov_empty  = coverage_ratio(empty_pcd, replace_pcd,  threshold)

            pipeline_coverage_gain = pipeline_cov_empty - var_cov_empty
            replace_coverage_gain  = replace_cov_empty  - var_cov_empty  # should be ~0

            row = dict(
                folder_name            = folder.name,
                env_type               = env_type(folder.name),
                variation              = var_path.name,
                # point counts
                n_ref                  = n_ref,
                n_var                  = len(var_pcd.points),
                n_pipeline             = n_pipeline,
                # primary metrics
                points_kept_pct        = points_kept_pct,
                points_updated_pct     = points_updated_pct,
                var_coverage_of_empty  = var_cov_empty      * 100,
                pipeline_coverage_gain = pipeline_coverage_gain * 100,
                replace_coverage_gain  = replace_coverage_gain  * 100,
                pipeline_total_cov     = pipeline_cov_empty * 100,
                replace_total_cov      = replace_cov_empty  * 100,
            )
            rows.append(row)

            print(
                f"  {var_path.name}"
                f"  kept={points_kept_pct:.1f}%"
                f"  updated={points_updated_pct:.1f}%"
                f"  pipeline_cov_gain={pipeline_coverage_gain*100:+.1f}%"
                f"  replace_cov_gain={replace_coverage_gain*100:+.1f}%"
            )

    if not rows:
        print("No results collected.")
        return pd.DataFrame(), pd.DataFrame()

    df = pd.DataFrame(rows)

    # ── aggregate by environment type ─────────────────────────────────────────
    metric_cols = [
        "points_kept_pct",
        "points_updated_pct",
        "var_coverage_of_empty",
        "pipeline_coverage_gain",
        "replace_coverage_gain",
        "pipeline_total_cov",
        "replace_total_cov",
    ]
    summary = (
        df.groupby("env_type")[metric_cols]
        .mean()
        .round(2)
        .reset_index()
        .rename(columns={"env_type": "environment_type"})
    )

    print("\n\n══ Summary by environment type ══")
    print(summary.to_string(index=False))

    return df, summary


# ── run ───────────────────────────────────────────────────────────────────────

DATASET_DIR      = Path(r"../../../datasets/V-Scan/data")
VOXEL_SIZE       = 0.05
THRESHOLD        = 0.05
CHECK_VISIBILITY = True
MAX_SCANS         = None  # set to an integer to limit the number of scans processed (for quick testing)

all_results, env_summary = evaluate_pipeline_vs_replace(
    dataset_dir      = DATASET_DIR,
    voxel_size       = VOXEL_SIZE,
    threshold        = THRESHOLD,
    check_visibility = CHECK_VISIBILITY,
    max_scans         = MAX_SCANS,
)

# Optional: save to CSV
# all_results.to_csv("eval_per_scan.csv", index=False)
# env_summary.to_csv("eval_per_env_type.csv", index=False)